# Data Quality — Validation Job Notebook

**Run this notebook daily (schedule in Fabric Job Scheduler).**

This notebook:
1. Loads all active checks from `check_registry`
2. Executes the DAX expression for each model
3. Compares all models against the configured baseline (`MODEL` row or `STATIC` numeric value)
4. Writes pass/fail results to `validation_results`
5. Shows a summary of failures (if any)

In [ ]:
# -- Configuration -------------------------------------------------------------
from datetime import datetime, timezone
import uuid
import time

# Preferred: load shared settings from Fabric config notebook.
try:
    ip = get_ipython()
    if ip is None:
        raise RuntimeError("IPython runtime not available")
    ip.run_line_magic("run", "data_quality_config_notebook")
except Exception:
    # Fallback keeps the notebook runnable if shared config execution is unavailable.
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"
    MAX_RETRY_ATTEMPTS = 3
    INITIAL_RETRY_DELAY = 2
    DAX_TIMEOUT_SECONDS = 60
    RUN_TABLE_MAINTENANCE = False
    MAINTENANCE_DAY_UTC = 6

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"
now_utc = datetime.now(timezone.utc)
RUN_DATE = now_utc.date()
RUN_TIMESTAMP = now_utc.replace(tzinfo=None)
RUN_ID = str(uuid.uuid4())

print(f"Run ID: {RUN_ID}")
print(f"Run Date: {RUN_DATE} (UTC)")
print(f"DAX timeout: {DAX_TIMEOUT_SECONDS}s")

In [ ]:
# -- Load Active Checks --------------------------------------------------------
checks_df = spark.sql(f"""
SELECT
    r.check_name,
    r.model_name,
    r.workspace_id,
    r.dataset_id,
    r.workspace_name,
    r.dataset_name,
    r.dax_expression,
    r.run_frequency,
    r.fail_delta_pct_threshold,
    r.is_baseline,
    COALESCE(c.baseline_mode, 'MODEL') AS baseline_mode,
    c.static_baseline_value
FROM {FULL_SCHEMA}.check_registry r
LEFT JOIN {FULL_SCHEMA}.check_baseline_config c
  ON r.check_name = c.check_name
WHERE r.is_active = true
ORDER BY r.check_name, r.model_name
""").toPandas()

print(f"\nLoaded {len(checks_df)} active check(s)")

# WARN AND EXIT IF REGISTRY IS EMPTY
if len(checks_df) == 0:
    print("\nWARNING: No active checks found in check_registry!")
    print("\n   Troubleshooting:")
    print("   1. Have you run data_quality_add_checks_notebook yet?")
    print("   2. Did you set is_active = false for all checks?")
    print("   3. Did you delete all rows from check_registry?")
    print("\n   Next steps:")
    print("   - Run data_quality_add_checks_notebook to register checks")
    print("   - OR run data_quality_delete_checks_notebook with DELETE_METHOD='soft' to reactivate")
    print(f"\n   Schema: {FULL_SCHEMA}")
    print("   If the schema or table doesn't exist, run data_quality_setup_notebook first.")
    raise RuntimeError("Empty check registry - no checks to validate")

# Validate required IDs
missing_ids = checks_df[
    checks_df["workspace_id"].isna()
    | checks_df["dataset_id"].isna()
    | (checks_df["workspace_id"].astype(str).str.strip() == "")
    | (checks_df["dataset_id"].astype(str).str.strip() == "")
]
if len(missing_ids) > 0:
    raise RuntimeError(
        "workspace_id/dataset_id are required for all active checks. "
        "Fix missing IDs using data_quality_add_checks_notebook."
    )

invalid_thresholds = checks_df[
    checks_df["fail_delta_pct_threshold"].isna()
    | (checks_df["fail_delta_pct_threshold"] < 0)
]
if len(invalid_thresholds) > 0:
    raise RuntimeError(
        "fail_delta_pct_threshold is required and must be >= 0 for all active checks. "
        "Fix missing thresholds using data_quality_update_checks_notebook.\n"
        + invalid_thresholds[["check_name", "model_name", "workspace_id", "dataset_id", "fail_delta_pct_threshold"]].head(20).to_string(index=False)
    )

# Load only the latest same-day result per (check_name, workspace_id, dataset_id)
# This prevents stale statuses from affecting ONCE_PER_DAY skip logic.
existing_results = spark.sql(f"""
WITH ranked AS (
    SELECT
        check_name,
        workspace_id,
        dataset_id,
        status,
        run_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY check_name, workspace_id, dataset_id
            ORDER BY run_timestamp DESC
        ) AS rn
    FROM {FULL_SCHEMA}.validation_results
    WHERE run_date = '{RUN_DATE}'
)
SELECT check_name, workspace_id, dataset_id, status, run_timestamp
FROM ranked
WHERE rn = 1
""").toPandas()

skip_count = 0
multi_count = 0
rerun_error_count = 0

if len(existing_results) > 0:
    # Load fresh run_frequency from check_registry (not stale from prior result joins)
    fresh_freq = spark.sql(f"""
        SELECT DISTINCT check_name, workspace_id, dataset_id, run_frequency
        FROM {FULL_SCHEMA}.check_registry
        WHERE is_active = true
    """).toPandas()
    existing_results = existing_results.merge(
        fresh_freq,
        on=["check_name", "workspace_id", "dataset_id"],
        how="left"
    )

    skippable = existing_results[
        (existing_results["run_frequency"] == "ONCE_PER_DAY")
        & (existing_results["status"] != "ERROR")
    ][["check_name", "workspace_id", "dataset_id"]].drop_duplicates()

    skip_count = len(skippable)
    multi_count = len(existing_results[existing_results["run_frequency"] == "MULTIPLE_PER_DAY"])
    rerun_error_count = len(existing_results[
        (existing_results["run_frequency"] == "ONCE_PER_DAY")
        & (existing_results["status"] == "ERROR")
    ])

    if skip_count > 0:
        print(f"  Skipping {skip_count} ONCE_PER_DAY check(s) already run today")
    if rerun_error_count > 0:
        print(f"  Re-running {rerun_error_count} ONCE_PER_DAY check(s) with prior ERROR status")
    if multi_count > 0:
        print(f"  Re-running {multi_count} MULTIPLE_PER_DAY check(s) (will add new rows)")

    checks_df = checks_df.merge(
        skippable.assign(should_skip=True),
        on=["check_name", "workspace_id", "dataset_id"],
        how="left"
    )
    checks_df["should_skip"] = checks_df["should_skip"].fillna(False)
else:
    checks_df["should_skip"] = False

# Materialize runnable set up front so preflight and baseline work only on rows that will execute.
checks_to_run_df = checks_df[~checks_df["should_skip"]].copy()

# Validate baseline mode + baseline requirements per runnable check_name.
baseline_counts_df = None
if len(checks_to_run_df) > 0:
    baseline_counts_df = (
        checks_to_run_df
        .groupby("check_name", as_index=False)
        .agg(
            active_row_count=("check_name", "count"),
            active_baseline_count=("is_baseline", lambda s: int((s == True).sum())),
            baseline_mode=("baseline_mode", "first"),
            static_baseline_value=("static_baseline_value", "first")
        )
    )

    baseline_counts_df["baseline_mode"] = baseline_counts_df["baseline_mode"].fillna("MODEL").astype(str).str.strip().str.upper()

    invalid_mode_df = baseline_counts_df[~baseline_counts_df["baseline_mode"].isin(["MODEL", "STATIC"])]
    if len(invalid_mode_df) > 0:
        raise RuntimeError(
            "Invalid baseline_mode found for runnable checks. Allowed values: MODEL, STATIC.\n"
            + invalid_mode_df[["check_name", "baseline_mode"]].head(20).to_string(index=False)
        )

    static_missing_df = baseline_counts_df[
        (baseline_counts_df["baseline_mode"] == "STATIC")
        & (baseline_counts_df["static_baseline_value"].isna())
    ]
    if len(static_missing_df) > 0:
        raise RuntimeError(
            "STATIC baseline_mode requires static_baseline_value for runnable checks.\n"
            + static_missing_df[["check_name", "baseline_mode", "static_baseline_value"]].head(20).to_string(index=False)
        )

    invalid_model_baselines_df = baseline_counts_df[
        (baseline_counts_df["baseline_mode"] == "MODEL")
        & (baseline_counts_df["active_baseline_count"] != 1)
    ]
    if len(invalid_model_baselines_df) > 0:
        raise RuntimeError(
            "Baseline invariant violation for runnable MODEL checks: each check_name must have exactly one active baseline row.\n"
            + invalid_model_baselines_df[["check_name", "active_row_count", "active_baseline_count"]].head(20).to_string(index=False)
        )

# Build baseline rows for MODEL-mode checks.
if len(checks_to_run_df) > 0:
    baseline_models = checks_to_run_df[
        (checks_to_run_df["baseline_mode"].astype(str).str.upper() == "MODEL")
        & (checks_to_run_df["is_baseline"] == True)
    ].copy()
else:
    baseline_models = checks_to_run_df.copy()

if len(baseline_models) > 0:
    baseline_models = baseline_models[[
        "check_name",
        "model_name",
        "workspace_name",
        "dataset_name",
        "dax_expression",
    ]]
    baseline_models.columns = [
        "check_name",
        "baseline_model_name",
        "baseline_workspace",
        "baseline_dataset",
        "baseline_dax",
    ]

# Persist for downstream execution cell
PRECOMPUTED_SKIPPED_COUNT = int(skip_count)
BASELINE_COUNTS_DF = baseline_counts_df

model_baseline_count = 0 if baseline_counts_df is None else int((baseline_counts_df["baseline_mode"] == "MODEL").sum())
static_baseline_count = 0 if baseline_counts_df is None else int((baseline_counts_df["baseline_mode"] == "STATIC").sum())

print(f"  Baseline checks in MODEL mode:  {model_baseline_count}")
print(f"  Baseline checks in STATIC mode: {static_baseline_count}")
print(f"  To process: {len(checks_to_run_df)} check(s)")

In [ ]:
# -- Execute DAX & Write Results with Batching, Retry, Timeout, & Type Safety --
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

# Validate sempy is available (early fail rather than runtime error later)
try:
    import sempy.fabric as sempy
except ImportError:
    raise RuntimeError(
        "sempy library not available. This notebook must run in Microsoft Fabric environment. "
        "Visit: https://learn.microsoft.com/en-us/python/api/sempy/"
    )

pass_count = 0
fail_count = 0
error_count = 0
skipped_count = PRECOMPUTED_SKIPPED_COUNT if "PRECOMPUTED_SKIPPED_COUNT" in locals() else 0
results_batch = []  # Accumulate results for batch write

# Helper: Execute a single DAX with timeout protection
# NOTE: This is best-effort timeout handling for notebook execution safety.
def evaluate_dax_with_timeout(dataset, workspace, dax, timeout_seconds=DAX_TIMEOUT_SECONDS):
    executor = ThreadPoolExecutor(max_workers=1)
    future = executor.submit(sempy.evaluate_dax, dataset=dataset, workspace=workspace, dax=dax)
    try:
        return future.result(timeout=timeout_seconds), None
    except FuturesTimeoutError:
        future.cancel()
        return None, f"DAX execution timed out after {timeout_seconds}s"
    except Exception as e:
        return None, str(e)
    finally:
        executor.shutdown(wait=False, cancel_futures=True)

# Helper: Retry with exponential backoff
def evaluate_dax_with_retry(dataset, workspace, dax, max_attempts=MAX_RETRY_ATTEMPTS, initial_delay=INITIAL_RETRY_DELAY):
    for attempt in range(1, max_attempts + 1):
        result_df, error_msg = evaluate_dax_with_timeout(dataset, workspace, dax)
        if error_msg is None:
            return result_df, None

        if attempt < max_attempts:
            delay = initial_delay ** attempt
            print(f"      Retry {attempt}/{max_attempts} failed, retrying in {delay}s...")
            time.sleep(delay)
        else:
            return None, error_msg
    return None, "Max retries exceeded"

# Helper: Validate numeric type
def extract_numeric_value(result_df, check_name):
    if result_df is None or len(result_df) == 0:
        return None, "DAX returned empty result"

    try:
        val = result_df.iloc[0, 0]

        if isinstance(val, (int, float)):
            return float(val), None
        if isinstance(val, str):
            try:
                return float(val), None
            except ValueError:
                return None, f"DAX returned non-numeric string: {val}"
        if val is None:
            return None, "DAX returned NULL"
        return None, f"DAX returned unsupported type: {type(val).__name__}"
    except Exception as e:
        return None, f"Failed to extract value: {str(e)[:200]}"

# Helper: MERGE batch for idempotent writes (safe on retry/restart)
def merge_results_batch(batch_rows):
    if not batch_rows:
        return

    batch_df = spark.createDataFrame(batch_rows)
    batch_df.createOrReplaceTempView("dq_results_batch")

    spark.sql(f"""
    MERGE INTO {FULL_SCHEMA}.validation_results t
    USING dq_results_batch s
      ON t.run_id = s.run_id
     AND t.check_name = s.check_name
     AND t.workspace_id = s.workspace_id
     AND t.dataset_id = s.dataset_id
    WHEN MATCHED THEN UPDATE SET
      t.run_date = s.run_date,
      t.run_timestamp = s.run_timestamp,
      t.model_name = s.model_name,
      t.workspace_id = s.workspace_id,
      t.dataset_id = s.dataset_id,
      t.workspace_name = s.workspace_name,
      t.dataset_name = s.dataset_name,
      t.result_value = s.result_value,
      t.baseline_model = s.baseline_model,
      t.baseline_value = s.baseline_value,
      t.absolute_delta = s.absolute_delta,
      t.delta_pct = s.delta_pct,
      t.fail_delta_pct_threshold = s.fail_delta_pct_threshold,
      t.status = s.status,
      t.error_message = s.error_message
    WHEN NOT MATCHED THEN INSERT (
      run_id, run_date, run_timestamp, check_name, model_name,
      workspace_id, dataset_id, workspace_name, dataset_name,
      result_value, baseline_model, baseline_value, absolute_delta,
      delta_pct, fail_delta_pct_threshold, status, error_message
    ) VALUES (
      s.run_id, s.run_date, s.run_timestamp, s.check_name, s.model_name,
      s.workspace_id, s.dataset_id, s.workspace_name, s.dataset_name,
      s.result_value, s.baseline_model, s.baseline_value, s.absolute_delta,
      s.delta_pct, s.fail_delta_pct_threshold, s.status, s.error_message
    )
    """)

if len(checks_to_run_df) == 0:
    print("\nNo checks require execution after skip rules.")

# Preflight: validate DAX shape and workspace/model reachability only for runnable rows
invalid_dax_count = 0
for _, row in checks_to_run_df.iterrows():
    dax_text = str(row["dax_expression"]).strip().upper()
    if not dax_text.startswith("EVALUATE"):
        invalid_dax_count += 1
        print(f"  DAX for check '{row['check_name']}' model '{row['model_name']}' does not start with EVALUATE")

if invalid_dax_count > 0:
    print(f"  Found {invalid_dax_count} potentially invalid DAX expression(s). Execution will continue.")

probe_targets = checks_to_run_df[["workspace_name", "dataset_name"]].drop_duplicates()
probe_failures = []
for _, target in probe_targets.iterrows():
    _, probe_err = evaluate_dax_with_retry(
        target["dataset_name"],
        target["workspace_name"],
        'EVALUATE ROW("probe", 1)',
        max_attempts=1,
        initial_delay=1
    )
    if probe_err:
        probe_failures.append(f"{target['workspace_name']} / {target['dataset_name']} -> {probe_err}")

if probe_failures:
    print("\nPreflight connectivity issues found:")
    for msg in probe_failures[:20]:
        print(f"   - {msg}")
    if len(probe_failures) > 20:
        print(f"   ... and {len(probe_failures) - 20} more")
    print("   Checks against these models may return ERROR.")

# Build baseline maps for both STATIC and MODEL modes.
baselines = {}  # {check_name: value}
baseline_model_map = {}  # {check_name: model label or STATIC_VALUE}

baseline_counts_df = BASELINE_COUNTS_DF if "BASELINE_COUNTS_DF" in locals() and BASELINE_COUNTS_DF is not None else pd.DataFrame()
if len(baseline_counts_df) > 0:
    static_rows = baseline_counts_df[baseline_counts_df["baseline_mode"] == "STATIC"]
    for _, row in static_rows.iterrows():
        baselines[row["check_name"]] = float(row["static_baseline_value"])
        baseline_model_map[row["check_name"]] = "STATIC_VALUE"

# Pre-compute baseline values for MODEL checks only.
if "baseline_models" in locals() and len(baseline_models) > 0:
    baseline_models_list = baseline_models.to_dict("records")

    for b in baseline_models_list:
        check_name = b["check_name"]
        workspace_name = b["baseline_workspace"]
        dataset_name = b["baseline_dataset"]
        dax_expression = b["baseline_dax"]

        result_df, error_msg = evaluate_dax_with_retry(dataset_name, workspace_name, dax_expression)
        if error_msg:
            print(f"  Baseline error for {check_name}: {error_msg}")
            baselines[check_name] = None
        else:
            value, extract_error = extract_numeric_value(result_df, check_name)
            if extract_error:
                print(f"  Baseline type error for {check_name}: {extract_error}")
                baselines[check_name] = None
            else:
                baselines[check_name] = value

        baseline_model_map[check_name] = b["baseline_model_name"]

# Process runnable checks only
num_checks = len(checks_to_run_df)
for row_idx, (_, row) in enumerate(checks_to_run_df.iterrows()):
    check_name = row["check_name"]
    model_name = row["model_name"]
    workspace_id = row["workspace_id"]
    dataset_id = row["dataset_id"]
    workspace_name = row["workspace_name"]
    dataset_name = row["dataset_name"]
    dax_expression = row["dax_expression"]
    fail_delta_pct_threshold = None if pd.isna(row["fail_delta_pct_threshold"]) else float(row["fail_delta_pct_threshold"])

    result_value = None
    error_msg = None
    status = "PASS"
    absolute_delta = None
    delta_pct = None
    baseline_value = baselines.get(check_name)

    if fail_delta_pct_threshold is None or fail_delta_pct_threshold < 0:
        status = "ERROR"
        error_msg = f"Invalid fail_delta_pct_threshold for check '{check_name}' / model '{model_name}'"
        error_count += 1
    else:
        # Execute DAX with retry logic
        result_df, error_msg = evaluate_dax_with_retry(dataset_name, workspace_name, dax_expression)

        if error_msg:
            status = "ERROR"
            error_count += 1
        else:
            value, extract_error = extract_numeric_value(result_df, check_name)
            if extract_error:
                result_value = None
                error_msg = extract_error
                status = "ERROR"
                error_count += 1
            else:
                result_value = value

                if baseline_value is not None and result_value is not None:
                    absolute_delta = result_value - baseline_value
                    if baseline_value != 0:
                        delta_pct = (absolute_delta / baseline_value) * 100
                        if abs(delta_pct) > fail_delta_pct_threshold:
                            status = "FAIL"
                            fail_count += 1
                        else:
                            status = "PASS"
                            pass_count += 1
                    else:
                        if result_value != 0:
                            status = "FAIL"
                            fail_count += 1
                        else:
                            status = "PASS"
                            pass_count += 1
                else:
                    status = "ERROR"
                    error_msg = f"Baseline is unavailable for check '{check_name}'"
                    error_count += 1

    results_batch.append({
        "run_id": RUN_ID,
        "run_date": RUN_DATE,
        "run_timestamp": RUN_TIMESTAMP,
        "check_name": check_name,
        "model_name": model_name,
        "workspace_id": workspace_id,
        "dataset_id": dataset_id,
        "workspace_name": workspace_name,
        "dataset_name": dataset_name,
        "result_value": result_value,
        "baseline_model": baseline_model_map.get(check_name),
        "baseline_value": baseline_value,
        "absolute_delta": absolute_delta,
        "delta_pct": delta_pct,
        "fail_delta_pct_threshold": fail_delta_pct_threshold,
        "status": status,
        "error_message": error_msg[:500] if error_msg else None
    })

    status_icon = "PASS" if status == "PASS" else "FAIL" if status == "FAIL" else "ERROR"
    print(
        f"  {status_icon:5} | {check_name:20} | {model_name:15} | "
        f"value={result_value} | threshold_pct={fail_delta_pct_threshold}"
    )

    if len(results_batch) >= 100 or row_idx == num_checks - 1:
        merge_results_batch(results_batch)
        results_batch = []

print("\nValidation run complete")
print(f"    PASS:    {pass_count}")
print(f"    FAIL:    {fail_count}")
print(f"    ERROR:   {error_count}")
print(f"    SKIPPED: {skipped_count}")

## Summary

In [ ]:
from IPython.display import display

# Show failures from THIS run only
failures = spark.sql(f"""
SELECT check_name, model_name, result_value, baseline_value, delta_pct, fail_delta_pct_threshold, status, error_message
FROM {FULL_SCHEMA}.validation_results
WHERE run_id = '{RUN_ID}' AND status != 'PASS'
ORDER BY check_name, model_name
""").toPandas()

if len(failures) > 0:
    print(f"\n⚠️  {len(failures)} failure(s) / error(s) in this run:\n")
    display(failures)
else:
    print(f"\n✅ No failures in this run")

print(f"\nRun ID: {RUN_ID}")
print(f"Completed at: {RUN_TIMESTAMP}")

## Table Maintenance

Automatic cleanup and optimization of Delta tables.

In [ ]:
import time

print("\n── Table Maintenance ──")

run_scheduled_maintenance = RUN_DATE.weekday() == MAINTENANCE_DAY_UTC
should_run_maintenance = RUN_TABLE_MAINTENANCE or run_scheduled_maintenance

if not should_run_maintenance:
    print("Skipping maintenance for this run (disabled and not scheduled day).")
    print(f"Scheduled weekday: {MAINTENANCE_DAY_UTC}, current weekday: {RUN_DATE.weekday()}")
else:
    if RUN_TABLE_MAINTENANCE:
        print("Maintenance mode: forced by RUN_TABLE_MAINTENANCE=True")
    else:
        print("Maintenance mode: scheduled weekday trigger")

    # OPTIMIZE validation_results — consolidate small files written incrementally
    print(f"OPTIMIZE {FULL_SCHEMA}.validation_results...")
    start = time.time()
    spark.sql(f"OPTIMIZE {FULL_SCHEMA}.validation_results")
    elapsed = time.time() - start
    print(f"  ✓ Optimized in {elapsed:.2f}s")

    # VACUUM validation_results — remove old file versions (retain 7 days)
    print(f"VACUUM {FULL_SCHEMA}.validation_results...")
    start = time.time()
    spark.sql(f"VACUUM {FULL_SCHEMA}.validation_results RETAIN 7 DAYS")
    elapsed = time.time() - start
    print(f"  ✓ Vacuumed in {elapsed:.2f}s")

    # ANALYZE TABLE — gather statistics for query optimization
    print(f"ANALYZE {FULL_SCHEMA}.validation_results...")
    start = time.time()
    spark.sql(f"ANALYZE TABLE {FULL_SCHEMA}.validation_results COMPUTE STATISTICS")
    elapsed = time.time() - start
    print(f"  ✓ Analyzed in {elapsed:.2f}s")

    print("\n✓  Maintenance complete")